# EA Sports FC 26 — Kaggle dataset explorer

Plays directly with the CSV we pulled (`data/raw/eafc26/eafc26-player-database/EAFC26.csv`),
**not** the DuckDB. 16,228 men + 1,645 women, 59 columns.

Run top-to-bottom once, then edit the filter cells and re-run. Nothing here
writes anything — it's just pandas over a file.

In [ ]:
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/raw/eafc26').exists())
CSV = ROOT / 'data/raw/eafc26/eafc26-player-database/EAFC26.csv'
raw = pd.read_csv(CSV)
print('shape:', raw.shape)
print('columns:', list(raw.columns))
raw.head()

## Men only (the WC-relevant set)
`men` is what every cell below uses. Club is the `Team` column; the 6 family
scores are `PAC SHO PAS DRI DEF PHY`.

In [ ]:
men = raw[raw['GENDER'] == 'M'].copy()
print(len(men), 'men')
men[['OVR', 'PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY', 'Age']].describe().round(1)

## What's in here? Positions, leagues, nations

In [ ]:
print('positions:'); print(men['Position'].value_counts())
print('\ntop 15 leagues:'); print(men['League'].value_counts().head(15))
print('\ntop 15 nations (EA spelling):'); print(men['Nation'].value_counts().head(15))

## Filter & rank — edit these and re-run 🐢
Set any of the filters to a value, or leave as `None` to ignore.

In [ ]:
POSITION = None      # e.g. 'ST', 'CB', 'GK'
NATION   = None      # e.g. 'Egypt', 'France', 'Holland'  (EA spelling)
TEAM     = None      # e.g. 'Liverpool', 'Real Madrid'
MIN_OVR  = 85        # minimum overall

f = men[men['OVR'] >= MIN_OVR]
if POSITION: f = f[f['Position'] == POSITION]
if NATION:   f = f[f['Nation'] == NATION]
if TEAM:     f = f[f['Team'] == TEAM]

cols = ['Name', 'OVR', 'Position', 'PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY',
        'Age', 'Nation', 'Team']
f.sort_values('OVR', ascending=False)[cols].head(25).reset_index(drop=True)

## Attribute leaders
Change `'PAC'` to any attribute column (e.g. `'Finishing'`, `'Strength'`, `'Dribbling'`).

In [ ]:
ATTR = 'PAC'
men.nlargest(15, ATTR)[['Name', ATTR, 'Position', 'Age', 'Nation', 'Team']].reset_index(drop=True)

## Head-to-head compare
Put any names in the list (exact EA spelling, accents and all).

In [ ]:
names = ['Mohamed Salah', 'Kylian Mbappé']
cols = ['Name', 'OVR', 'PAC', 'SHO', 'PAS', 'DRI', 'DEF', 'PHY',
        'Dribbling', 'Finishing', 'Vision', 'Stamina']
men[men['Name'].isin(names)][cols].set_index('Name').T

## PlayStyles
Reuses our loader's parser to explode the `play style` column into clean
(player, playstyle, tier) rows — then you can ask "who has X?".

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'src/load/v2_ingest'))
from ingest_ea_fc26 import parse_ea

pdf, sdf, _ = parse_ea(CSV)   # pdf = cleaned players, sdf = exploded playstyles
print(len(sdf), 'playstyle rows;', sdf['playstyle'].nunique(), 'distinct styles')
sdf['playstyle'].value_counts().head(20)

In [ ]:
# who has a given PlayStyle? (try 'Finesse Shot', 'Aerial Fortress', 'Rapid')
STYLE = 'Finesse Shot'
ids = sdf.loc[sdf['playstyle'] == STYLE, 'ea_id']
pdf[pdf['ea_id'].isin(ids)].sort_values('ovr', ascending=False)[
    ['name', 'ovr', 'position', 'nation_name', 'club']].head(15).reset_index(drop=True)

## Your turn
`men` (raw columns) and `pdf` (cleaned, snake_case) are both in memory. Slice away.

In [ ]:
# scratch
men.groupby('Position')['OVR'].mean().round(1).sort_values(ascending=False)